In [22]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['AppleGothic']


## 실습 1. C-MAPSS 데이터 열고 구조 확인
### 목표
강사가 준 엔진 데이터를 직접 열어 크기와 열 구성을 파악
### 단계
- pandas를 불러와 샘플 파일을 데이터프레임으로 열기
- 크기 속성으로 행·열 개수를, 열 이름 속성으로 컬럼을 확인
- 앞부분을 미리 보며 값의 모양을 눈으로 익히기
### 예상 결과
행 천여 개·스물여섯 열, 엔진번호·작동회차·센서 열이 보임

In [23]:
df = pd.read_csv("data/20_cmapss_fd001_sample.csv")
print(df.shape)
print(df.columns)

(1116, 26)
Index(['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3', 's_1',
       's_2', 's_3', 's_4', 's_5', 's_6', 's_7', 's_8', 's_9', 's_10', 's_11',
       's_12', 's_13', 's_14', 's_15', 's_16', 's_17', 's_18', 's_19', 's_20',
       's_21'],
      dtype='str')


## 실습 2. 엔진 단위와 작동 길이 파악
### 목표
데이터에 여러 엔진이 섞여 있음을 확인하고 엔진별 작동 길이를 파악
### 단계
- 엔진번호의 고유값 개수로 엔진이 몇 대인지 세기
- 엔진번호별 행 수를 세어 각 엔진의 작동 회차 길이를 구하기
- 번호 순으로 정렬해 엔진마다 길이가 다름을 확인
### 예상 결과
엔진 다섯 대, 엔진별 회차 수가 179~287로 서로 다름

In [24]:
print("엔진 대수 :", df["unit_nr"].nunique())

print("엔진별 사이클 기록 갯수")
print(df["unit_nr"].value_counts())

엔진 대수 : 5
엔진별 사이클 기록 갯수
unit_nr
2    287
5    269
1    192
4    189
3    179
Name: count, dtype: int64


## 실습 3. 요약 통계로 센서 범위 읽기
### 목표
요약 통계로 주요 센서의 평균과 값 범위를 수치로 파악
### 단계
- 2번 엔진을 골라 세 센서의 요약 통계를 한 번에 보기
- 평균으로 대략 크기를, 흔들림 폭으로 변동 정도를 읽기
- 최소·최대로 각 센서의 값 범위가 서로 다름을 확인
### 예상 결과
세 센서의 평균이 약 641.8·47.7·517.4로 크기가 제각각

In [25]:
engine = df[df["unit_nr"] == 2]
print(engine[["s_2", "s_12", "s_18"]].describe())

              s_2        s_12    s_18
count  287.000000  287.000000   287.0
mean   641.752997  517.430105  2388.0
std      0.486257    2.146613     0.0
min    640.460000  512.860000  2388.0
25%    641.390000  515.655000  2388.0
50%    641.700000  517.230000  2388.0
75%    642.140000  519.155000  2388.0
max    642.870000  521.960000  2388.0


## 실습 4. 변하는 센서 선별과 결측 확인
### 목표
거의 변하지 않는 센서를 빼고 분석에 쓸 센서를 직접 선별
### 단계
- 여러 센서의 흔들림 폭을 비교해 0에 가까운 것은 제외 대상으로 표시
- 뚜렷이 변하는 센서를 분석 대상으로 남기기
- 결측 합계를 세어 빈 값이 없는지 확인
### 예상 결과
첫 두 센서는 값이 늘 같아 제외, 뒤 센서는 변함, 결측 없음

In [26]:
sensors = ["s_1", "s_3", "s_8", "s_10", "s_13"]

for sensor in sensors:
    print(sensor, round(df[sensor].std(), 3))
print("-" * 20)
print("결측 합계: ", df.isna().sum().sum())

s_1 0.0
s_3 4.573
s_8 0.099
s_10 0.0
s_13 0.076
--------------------
결측 합계:  0


## 실습 5. 엔진 하나 골라 첫·끝 추세 읽기
### 목표
엔진 하나를 골라 회차 순으로 정렬하고 첫·끝 값으로 추세 방향을 읽기
### 단계
- 2번 엔진만 골라 작동 회차 순으로 정렬
- 상승 센서와 하강 센서의 첫 회차 값과 마지막 값을 꺼내기
- 첫 값과 끝 값을 비교해 상승인지 하강인지 판정
### 예상 결과
한 센서는 47.30에서 48.18로 상승, 다른 센서는 하강

## 실습 6. 엔진 센서 시계열 그래프 그리기
### 목표
정렬한 엔진 데이터로 회차를 가로축, 센서를 세로축으로 선 그래프 그리기
### 단계
- 상승 센서를 선 그래프로 그리고 제목·축 이름 붙이기
- 하강 센서도 같은 방식으로 그려 방향을 비교
- 두 그래프의 기울기 방향으로 추세를 눈으로 확인
### 예상 결과
한 센서는 우상향, 다른 센서는 우하향 곡선

## 실습 7. 여러 센서 비교와 범위 문제
### 목표
여러 센서를 함께 그려 비교하고 값 범위가 다를 때의 문제를 체험
### 단계
- 크기가 비슷한 두 센서를 함께 그리고 범례를 붙여 비교
- 크기가 크게 다른 두 센서를 일부러 겹쳐 그리기
- 두 센서의 범위를 숫자로 확인해 작은 값이 왜 안 보이는지 이해
### 예상 결과
약 8100 센서에 가려 약 38 센서가 평평하게 보임

## 실습 8. 데이터 구조 미니 리포트
### 목표
오늘 확인한 크기·엔진·센서 특성을 짧은 리포트로 자동 정리
### 단계
- 전체 크기와 엔진 대수를 한 줄로 출력
- 대상 엔진의 작동 회차 수와 결측 유무를 정리
- 주목 센서 세 개의 첫·끝 방향을 반복문으로 표처럼 정리
### 예상 결과
크기·엔진 수·회차·결측·센서 방향이 담긴 짧은 명세 리포트